In [2]:
#IMPORTS 
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import shap

from scipy import stats
from statsmodels.stats.multitest import multipletests

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.metrics import (
    roc_auc_score,
    roc_curve,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")
matplotlib.use("Agg")

In [3]:
#CONFIGURATION

DATA_PATH = "../data/alzheimers_disease_data.csv"

# Second dataset is optional for now.
# Fill this in later once the second dataset is chosen.
DATA_PATH_2 = None
SECOND_DATASET_NAME = None  # Example: "OASIS-2" or "Global Alzheimer Kaggle"

TARGET = "Diagnosis"
RANDOM_STATE = 42

CV_FOLDS = 5
N_BOOT = 1000
N_PERM = 200

OUTPUT_DIR = Path("results")
OUTPUT_DIR.mkdir(exist_ok=True)

FAIRNESS_COLS = ["Gender", "AgeGroup"]

In [4]:
#FEATURE SETS

FEATURE_SETS = {
    "FS1_lifestyle": [
        "Age", "Gender", "Ethnicity", "EducationLevel", "BMI",
        "Smoking", "AlcoholConsumption", "PhysicalActivity",
        "DietQuality", "SleepQuality",
    ],

    "FS2_lifestyle_medical": [
        "Age", "Gender", "Ethnicity", "EducationLevel", "BMI",
        "Smoking", "AlcoholConsumption", "PhysicalActivity",
        "DietQuality", "SleepQuality",
        "FamilyHistoryAlzheimers", "CardiovascularDisease", "Diabetes",
        "Depression", "HeadInjury", "Hypertension",
        "SystolicBP", "DiastolicBP",
        "CholesterolTotal", "CholesterolLDL",
        "CholesterolHDL", "CholesterolTriglycerides",
    ],

    "FS3_full": [
    "Age", "Gender", "Ethnicity", "EducationLevel", "BMI",
    "Smoking", "AlcoholConsumption", "PhysicalActivity",
    "DietQuality", "SleepQuality",
    "FamilyHistoryAlzheimers", "CardiovascularDisease", "Diabetes",
    "Depression", "HeadInjury", "Hypertension",
    "SystolicBP", "DiastolicBP",
    "CholesterolTotal", "CholesterolLDL",
    "CholesterolHDL", "CholesterolTriglycerides",
    "MMSE", "FunctionalAssessment",
    "MemoryComplaints", "Confusion", "BehavioralProblems",
    "ADL", "Disorientation", "PersonalityChanges",
    "DifficultyCompletingTasks", "Forgetfulness",
    ],
}

In [5]:
#LOAD PRIMARY DATASET
def load_primary_data(path: str) -> pd.DataFrame:

    df = pd.read_csv(path)

    df = df.drop(columns=["PatientID", "DoctorInCharge"], errors="ignore")

    if "Age" in df.columns:
        df["AgeGroup"] = pd.cut(
            df["Age"],
            bins=[0, 59, 69, 79, 120],
            labels=["<60", "60-69", "70-79", "80+"],
            right=True
        )

    return df

#Print and return class distribution
def inspect_class_balance(y: pd.Series) -> dict:

    counts = y.value_counts().sort_index()
    ratio = counts.min() / counts.max()

    print("\nClass balance")
    print("-" * 40)

    for cls, n in counts.items():
        print(f"Class {cls}: {n} ({100 * n / len(y):.1f}%)")

    print(f"Minority / majority ratio: {ratio:.3f}")

    return {
        "counts": counts.to_dict(),
        "ratio": round(float(ratio), 4),
    }

In [6]:
#TRAIN TEST SPLIT 
#make statified split 85/15
def make_splits(df: pd.DataFrame, features: list):
    model_features = [f for f in features if f not in ["AgeGroup", TARGET]]

    X = df[model_features].copy()
    y = df[TARGET].copy()

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.15,
        stratify=y,
        random_state=RANDOM_STATE,
    )

    return X_train, X_test, y_train, y_test

In [7]:
#MODELS
def get_models(y_train: pd.Series):

    n_negative = (y_train == 0).sum()
    n_positive = (y_train == 1).sum()
    scale_pos_weight = n_negative / n_positive

    models = {
        "LogisticRegression": {
            "estimator": Pipeline([
                ("scaler", StandardScaler()),
                ("model", LogisticRegression(
                    class_weight="balanced",
                    max_iter=1000,
                    random_state=RANDOM_STATE,
                ))
            ]),
            "param_grid": {
                "model__C": [0.01, 0.1, 1, 10],
                "model__solver": ["lbfgs", "saga"],
            },
        },

        "RandomForest": {
            "estimator": RandomForestClassifier(
                class_weight="balanced",
                random_state=RANDOM_STATE,
                n_jobs=-1,
            ),
            "param_grid": {
                "n_estimators": [100, 300],
                "max_depth": [None, 10, 20],
                "min_samples_leaf": [1, 2],
            },
        },

        "XGBoost": {
            "estimator": XGBClassifier(
                eval_metric="logloss",
                random_state=RANDOM_STATE,
                scale_pos_weight=scale_pos_weight,
                n_jobs=-1,
            ),
            "param_grid": {
                "learning_rate": [0.01, 0.1],
                "max_depth": [3, 5],
                "n_estimators": [100, 300],
                "subsample": [0.8, 1.0],
                "colsample_bytree": [0.8, 1.0],
            },
        },
    }

    return models

In [8]:
#TUNING AND EVALUATION
def tune_model(estimator, param_grid, X_train, y_train):
    cv = StratifiedKFold(
        n_splits=CV_FOLDS,
        shuffle=True,
        random_state=RANDOM_STATE,
    )

    grid = GridSearchCV(
        estimator=estimator,
        param_grid=param_grid,
        cv=cv,
        scoring="roc_auc",
        n_jobs=-1,
        refit=True,
    )

    grid.fit(X_train, y_train)

    return grid.best_estimator_, grid.best_params_, grid.best_score_

#evaluate model on a held out set
def evaluate_model(model, X_test, y_test):

    prob = model.predict_proba(X_test)[:, 1]
    pred = (prob >= 0.5).astype(int)

    return {
        "roc_auc": roc_auc_score(y_test, prob),
        "precision": precision_score(y_test, pred, zero_division=0),
        "recall": recall_score(y_test, pred, zero_division=0),
        "f1": f1_score(y_test, pred, zero_division=0),
        "prob": prob,
        "pred": pred,
    }

In [9]:
#BOOTSTRAP CONFIDENCE INTERVALS
#percentile bootstrap interval 
def bootstrap_ci(y_true, values, metric_fn, n_boot=N_BOOT, ci=0.95):

    rng = np.random.default_rng(RANDOM_STATE)

    y_true = np.asarray(y_true)
    values = np.asarray(values)

    boot_scores = []
    n = len(y_true)

    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)

        try:
            score = metric_fn(y_true[idx], values[idx])
            boot_scores.append(score)
        except Exception:
            continue

    lower = np.percentile(boot_scores, (1 - ci) / 2 * 100)
    upper = np.percentile(boot_scores, (1 + ci) / 2 * 100)

    return round(float(lower), 4), round(float(upper), 4)

In [10]:
#ROC PLOTTING 
#save ROC plot for feature sets 
def plot_roc_curves(roc_data, feature_set):

    fig, ax = plt.subplots(figsize=(6, 5))

    ax.plot([0, 1], [0, 1], linestyle="--", linewidth=1, label="Chance")

    for model_name, data in roc_data.items():
        fpr, tpr, auc_value = data
        ax.plot(
            fpr,
            tpr,
            linewidth=2,
            label=f"{model_name} AUC = {auc_value:.3f}"
        )

    ax.set_xlabel("False positive rate")
    ax.set_ylabel("True positive rate")
    ax.set_title(f"ROC curves: {feature_set}")
    ax.legend(loc="lower right")
    plt.tight_layout()

    path = OUTPUT_DIR / f"roc_{feature_set}.png"
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()

    print(f"ROC plot saved: {path}")

In [11]:
#FAIRNESS/SUBGROUP METRICS
#TPR, FPR, subgroup ROC-AUC, positive prediction rate
def fairness_metrics(df_test, y_true, prob, pred, sensitive_col):

    rows = []

    for group in df_test[sensitive_col].dropna().unique():
        mask = df_test[sensitive_col] == group

        if mask.sum() < 10:
            continue

        y_g = np.asarray(y_true)[mask.values]
        prob_g = prob[mask.values]
        pred_g = pred[mask.values]

        tn, fp, fn, tp = confusion_matrix(y_g, pred_g, labels=[0, 1]).ravel()

        tpr = tp / (tp + fn) if (tp + fn) > 0 else np.nan
        fpr = fp / (fp + tn) if (fp + tn) > 0 else np.nan
        auc = roc_auc_score(y_g, prob_g) if len(np.unique(y_g)) > 1 else np.nan

        rows.append({
            "sensitive_col": sensitive_col,
            "group": group,
            "n": int(mask.sum()),
            "positive_rate": round(float(pred_g.mean()), 4),
            "tpr": round(float(tpr), 4) if not np.isnan(tpr) else np.nan,
            "fpr": round(float(fpr), 4) if not np.isnan(fpr) else np.nan,
            "roc_auc": round(float(auc), 4) if not np.isnan(auc) else np.nan,
        })

    result = pd.DataFrame(rows)

    if result.empty:
        return result

    dp_diff = result["positive_rate"].max() - result["positive_rate"].min()
    tpr_diff = result["tpr"].max() - result["tpr"].min()
    fpr_diff = result["fpr"].max() - result["fpr"].min()

    result["demographic_parity_difference"] = round(float(dp_diff), 4)
    result["equalized_odds_difference"] = round(float(max(tpr_diff, fpr_diff)), 4)

    return result

In [15]:
#reload the data to ensure clean data
df = load_primary_data(DATA_PATH)

print("df loaded")
print("df shape:", df.shape)
display(df.head())

df loaded
df shape: (2149, 34)


,Age,Gender,Ethnicity,EducationLevel,BMI,Smoking,AlcoholConsumption,PhysicalActivity,DietQuality,SleepQuality,...,MemoryComplaints,BehavioralProblems,ADL,Confusion,Disorientation,PersonalityChanges,DifficultyCompletingTasks,Forgetfulness,Diagnosis,AgeGroup
0,73,0,0,2,22.927749,0,13.297218,6.327112,1.347214,9.025679,...,0,0,1.725883,0,0,0,1,0,0,70-79
1,89,0,0,0,26.827681,0,4.542524,7.619885,0.518767,7.151293,...,0,0,2.592424,0,0,0,0,1,0,80+
2,73,0,3,1,17.795882,0,19.555085,7.844988,1.826335,9.673574,...,0,0,7.119548,0,1,0,1,0,0,70-79
3,74,1,0,1,33.800817,1,12.209266,8.428001,7.435604,8.392554,...,0,1,6.481226,0,0,0,0,0,0,70-79
4,89,0,0,0,20.716974,0,18.454356,6.310461,0.795498,5.597238,...,0,0,0.014691,0,0,1,1,0,0,80+


In [13]:
#PRIMARY MODELLING LOOP 
all_results = []
all_fairness = []
fitted_models = {}
roc_predictions = {}

for feature_set, features in FEATURE_SETS.items():
    print("\n" + "=" * 70)
    print(f"Feature set: {feature_set} ({len(features)} features)")
    print("=" * 70)

    X_train, X_test, y_train, y_test = make_splits(df, features)
    df_test = df.loc[X_test.index].copy()

    print(f"Train shape:      {X_train.shape}")
    print(f"Test shape:       {X_test.shape}")

    models = get_models(y_train)
    roc_data = {}

    for model_name, config in models.items():
        print(f"\nTuning model: {model_name}")

        final_model, best_params, best_cv_auc = tune_model(
            estimator=config["estimator"],
            param_grid=config["param_grid"],
            X_train=X_train,
            y_train=y_train,
        )

        metrics = evaluate_model(final_model, X_test, y_test)

        prob = metrics["prob"]
        pred = metrics["pred"]

        auc_low, auc_high = bootstrap_ci(
            y_test.values,
            prob,
            roc_auc_score,
        )

        precision_low, precision_high = bootstrap_ci(
            y_test.values,
            pred,
            lambda yt, yp: precision_score(yt, yp, zero_division=0),
        )

        recall_low, recall_high = bootstrap_ci(
            y_test.values,
            pred,
            lambda yt, yp: recall_score(yt, yp, zero_division=0),
        )

        f1_low, f1_high = bootstrap_ci(
            y_test.values,
            pred,
            lambda yt, yp: f1_score(yt, yp, zero_division=0),
        )

        print(f"Best CV ROC-AUC: {best_cv_auc:.4f}")
        print(f"Test ROC-AUC:    {metrics['roc_auc']:.4f} [{auc_low}, {auc_high}]")
        print(f"Precision:       {metrics['precision']:.4f}")
        print(f"Recall:          {metrics['recall']:.4f}")
        print(f"F1-score:        {metrics['f1']:.4f}")
        print(f"Best params:     {best_params}")

        fpr, tpr, _ = roc_curve(y_test, prob)
        roc_data[model_name] = (fpr, tpr, metrics["roc_auc"])

        fitted_models[(feature_set, model_name)] = {
            "model": final_model,
            "best_params": best_params,
            "best_cv_auc": best_cv_auc,
            "X_train": X_train,
            "X_test": X_test,
            "y_train": y_train,
            "y_test": y_test,
            "prob": prob,
            "pred": pred,
        }

        roc_predictions[(feature_set, model_name)] = {
            "y_test": y_test,
            "prob": prob,
        }

        all_results.append({
            "feature_set": feature_set,
            "model": model_name,
            "best_params": str(best_params),
            "best_cv_auc": round(float(best_cv_auc), 4),
            "roc_auc": round(float(metrics["roc_auc"]), 4),
            "auc_ci_low": auc_low,
            "auc_ci_high": auc_high,
            "precision": round(float(metrics["precision"]), 4),
            "precision_ci_low": precision_low,
            "precision_ci_high": precision_high,
            "recall": round(float(metrics["recall"]), 4),
            "recall_ci_low": recall_low,
            "recall_ci_high": recall_high,
            "f1": round(float(metrics["f1"]), 4),
            "f1_ci_low": f1_low,
            "f1_ci_high": f1_high,
        })

        for sensitive_col in FAIRNESS_COLS:
            if sensitive_col in df_test.columns:
                fair_df = fairness_metrics(
                    df_test=df_test,
                    y_true=y_test,
                    prob=prob,
                    pred=pred,
                    sensitive_col=sensitive_col,
                )

                if not fair_df.empty:
                    fair_df["feature_set"] = feature_set
                    fair_df["model"] = model_name
                    all_fairness.append(fair_df)

    plot_roc_curves(roc_data, feature_set)


Feature set: FS1_lifestyle (10 features)
Train shape:      (1826, 10)
Test shape:       (323, 10)

Tuning model: LogisticRegression
Best CV ROC-AUC: 0.5169
Test ROC-AUC:    0.5501 [0.487, 0.6123]
Precision:       0.3636
Recall:          0.4912
F1-score:        0.4179
Best params:     {'model__C': 0.01, 'model__solver': 'saga'}

Tuning model: RandomForest
Best CV ROC-AUC: 0.4958
Test ROC-AUC:    0.4691 [0.4027, 0.5363]
Precision:       0.2542
Recall:          0.1316
F1-score:        0.1734
Best params:     {'max_depth': 10, 'min_samples_leaf': 1, 'n_estimators': 300}

Tuning model: XGBoost
Best CV ROC-AUC: 0.5011
Test ROC-AUC:    0.5089 [0.4408, 0.5735]
Precision:       0.3529
Recall:          0.4211
F1-score:        0.3840
Best params:     {'colsample_bytree': 0.8, 'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 300, 'subsample': 0.8}
ROC plot saved: results/roc_FS1_lifestyle.png

Feature set: FS2_lifestyle_medical (22 features)
Train shape:      (1826, 22)
Test shape:       (32

In [14]:
#SAVE RESULTS
results_df = pd.DataFrame(all_results)
results_df.to_csv(OUTPUT_DIR / "performance_results.csv", index=False)

print("Primary performance results:")
display(results_df)

if all_fairness:
    fairness_df = pd.concat(all_fairness, ignore_index=True)
    fairness_df.to_csv(OUTPUT_DIR / "fairness_results.csv", index=False)

    print("\nFairness/subgroup results:")
    display(fairness_df.head())
else:
    fairness_df = pd.DataFrame()
    print("No fairness results were generated.")

Primary performance results:


,feature_set,model,best_params,best_cv_auc,roc_auc,auc_ci_low,auc_ci_high,precision,precision_ci_low,precision_ci_high,recall,recall_ci_low,recall_ci_high,f1,f1_ci_low,f1_ci_high
0,FS1_lifestyle,LogisticRegression,"{'model__C': 0.01, 'model__solver': 'saga'}",0.5169,0.5501,0.4870,0.6123,0.3636,0.2925,0.4391,0.4912,0.4018,0.5872,0.4179,0.3422,0.4875
1,FS1_lifestyle,RandomForest,"{'max_depth': 10, 'min_samples_leaf': 1, 'n_es...",0.4958,0.4691,0.4027,0.5363,0.2542,0.1429,0.3621,0.1316,0.0748,0.1964,0.1734,0.0983,0.2486
2,FS1_lifestyle,XGBoost,"{'colsample_bytree': 0.8, 'learning_rate': 0.1...",0.5011,0.5089,0.4408,0.5735,0.3529,0.2687,0.4348,0.4211,0.3303,0.5146,0.3840,0.3000,0.4571
3,FS2_lifestyle_medical,LogisticRegression,"{'model__C': 10, 'model__solver': 'lbfgs'}",0.5359,0.5480,0.4818,0.6179,0.3598,0.2857,0.4334,0.5175,0.4215,0.6132,0.4245,0.3500,0.5000
4,FS2_lifestyle_medical,RandomForest,"{'max_depth': None, 'min_samples_leaf': 2, 'n_...",0.5218,0.5080,0.4446,0.5746,0.3333,0.0833,0.6112,0.0351,0.0081,0.0700,0.0635,0.0149,0.1231
5,FS2_lifestyle_medical,XGBoost,"{'colsample_bytree': 0.8, 'learning_rate': 0.1...",0.5278,0.4433,0.3804,0.5100,0.2959,0.2075,0.3900,0.2544,0.1818,0.3362,0.2736,0.1978,0.3524
6,FS3_full,LogisticRegression,"{'model__C': 0.1, 'model__solver': 'lbfgs'}",0.9002,0.8936,0.8530,0.9294,0.6759,0.6014,0.7466,0.8596,0.7840,0.9194,0.7568,0.6937,0.8095
7,FS3_full,RandomForest,"{'max_depth': 10, 'min_samples_leaf': 2, 'n_es...",0.9511,0.9505,0.9167,0.9782,0.9464,0.9010,0.9825,0.9298,0.8780,0.9720,0.9381,0.9018,0.9675
8,FS3_full,XGBoost,"{'colsample_bytree': 1.0, 'learning_rate': 0.1...",0.9557,0.9436,0.9051,0.9747,0.9464,0.9016,0.9826,0.9298,0.8793,0.9706,0.9381,0.9038,0.9662



Fairness/subgroup results:


,sensitive_col,group,n,positive_rate,tpr,fpr,roc_auc,demographic_parity_difference,equalized_odds_difference,feature_set,model
0,Gender,1,163,0.4417,0.6038,0.3636,0.6844,0.0708,0.2223,FS1_lifestyle,LogisticRegression
1,Gender,0,160,0.5125,0.3934,0.5859,0.4196,0.0708,0.2223,FS1_lifestyle,LogisticRegression
2,AgeGroup,60-69,108,0.5926,0.6757,0.5493,0.5672,0.2683,0.3868,FS1_lifestyle,LogisticRegression
3,AgeGroup,80+,111,0.3243,0.2889,0.3485,0.5407,0.2683,0.3868,FS1_lifestyle,LogisticRegression
4,AgeGroup,70-79,104,0.5192,0.5625,0.5000,0.5716,0.2683,0.3868,FS1_lifestyle,LogisticRegression
